In [1]:
!pip install -q --upgrade \
    transformers huggingface_hub \
    langchain langchain_huggingface \
    bitsandbytes faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 11.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 31.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.

Dự án về AI hỏi đáp pháp luật

In [5]:
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
import faiss
from sentence_transformers import SentenceTransformer

In [6]:
query_1 = "Các mức phạt khi điều khiển xe máy mà đã uống rượu"
context_1 = """
Điều 13 100/2015/QH13 tố tụng hình sự Phạm tội do dùng rượu, bia hoặc chất kích thích mạnh khác Người phạm tội trong tình trạng mất khả năng nhận thức hoặc khả năng điều khiển hành vi của mình do dùng rượu, bia hoặc chất kích thích mạnh khác, thì vẫn phải chịu trách nhiệm hình sự.
Điều 131 100/2015/QH13 tố tụng hình sự Tội xúi giục hoặc giúp người khác tự sát 1. Người nào thực hiện một trong các hành vi sau đây, thì bị phạt cải tạo không giam giữ đến 03 năm hoặc phạt tù từ 06 tháng đến 03 năm: a) Kích động, dụ dỗ, thúc đẩy người khác tự tước đoạt tính mạng của họ; b) Tạo điều kiện vật chất hoặc tinh thần cho người khác tự tước đoạt tính mạng của họ. 2. Phạm tội làm 02 người trở lên tự sát, thì bị phạt tù từ 02 năm đến 07 năm.
Điều 132 100/2015/QH13 tố tụng hình sự Tội không cứu giúp người đang ở trong tình trạng nguy hiểm đến tính mạng 1. Người nào thấy người khác đang ở trong tình trạng nguy hiểm đến tính mạng, tuy có điều kiện mà không cứu giúp dẫn đến hậu quả người đó chết, thì bị phạt cảnh cáo, phạt cải tạo không giam giữ đến 02 năm hoặc phạt tù từ 03 tháng đến 02 năm. 2. Phạm tội thuộc một trong các trường hợp sau đây, thì bị phạt tù từ 01 năm đến 05 năm: a) Người không cứu giúp là người đã vô ý gây ra tình trạng nguy hiểm; b) Người không cứu giúp là người mà theo pháp luật hay nghề nghiệp có nghĩa vụ phải cứu giúp. 3. Phạm tội dẫn đến hậu quả 02 người trở lên chết, thì bị phạt tù từ 03 năm đến 07 năm. 4. Người phạm tội còn có thể bị cấm đảm nhiệm chức vụ, cấm hành nghề hoặc làm công việc nhất định từ 01 năm đến 05 năm.
Điều 26 100/2015/QH13 tố tụng hình sự Thi hành mệnh lệnh của người chỉ huy hoặc của cấp trên Người thực hiện hành vi gây thiệt hại trong khi thi hành mệnh lệnh của người chỉ huy hoặc của cấp trên trong lực lượng vũ trang nhân dân để thực hiện nhiệm vụ quốc phòng, an ninh nếu đã thực hiện đầy đủ quy trình báo cáo người ra mệnh lệnh nhưng người ra mệnh lệnh vẫn yêu cầu chấp hành mệnh lệnh đó, thì không phải chịu trách nhiệm hình sự. Trong trường hợp này người ra mệnh lệnh phải chịu trách nhiệm hình sự. Quy định này không áp dụng đối với các trường hợp quy định tại khoản 2 Điều 421, khoản 2 Điều 422 và khoản 2 Điều 423 của Bộ luật này.
Điều 30 100/2015/QH13 tố tụng hình sự Khái niệm hình phạt Hình phạt là biện pháp cưỡng chế nghiêm khắc nhất của Nhà nước được quy định trong Bộ luật này, do Tòa án quyết định áp dụng đối với người hoặc pháp nhân thương mại phạm tội nhằm tước bỏ hoặc hạn chế quyền, lợi ích của người, pháp nhân thương mại đó.
"""
answer_1 = """
# Xử lý khi người điều khiển xe máy đã uống rượu 🍺  

## Căn cứ pháp lý  

**Điều 13 – Bộ luật Hình sự 100/2015/QH13:**  
Người phạm tội trong tình trạng mất khả năng nhận thức hoặc khả năng điều khiển hành vi của mình do dùng rượu, bia hoặc chất kích thích mạnh khác **vẫn phải chịu trách nhiệm hình sự**.  

**Điều 132 – Bộ luật Hình sự 100/2015/QH13:**  
Người **không cứu giúp người đang ở trong tình trạng nguy hiểm đến tính mạng**, nếu hậu quả dẫn đến người đó chết, **sẽ bị phạt tù từ 01 năm đến 05 năm**.  

## Mức xử lý cụ thể  

Nếu **người điều khiển xe máy đã uống rượu** và **không cứu giúp người đang ở trong tình trạng nguy hiểm đến tính mạng**, thì sẽ **bị phạt tù từ 01 năm đến 05 năm**.  
"""
query_2 = """Hãy giải thích chi tiết về Quốc Hội Việt Nam?"""
answer_2 = """
# Quốc hội Việt Nam 🇻🇳  

## Vị trí và vai trò  

**Quốc hội Việt Nam** là **cơ quan quyền lực cao nhất của Nhà nước** và là **cơ quan đại biểu cao nhất của nhân dân**.  

Theo **Điều 82** và **Điều 83** của **Hiến pháp năm 1992 và 1980**, Quốc hội có vai trò **quyết định những chính sách cơ bản** về:  
- Đối nội và đối ngoại  
- Nhiệm vụ kinh tế – xã hội, quốc phòng – an ninh của đất nước  
- Nguyên tắc chủ yếu về tổ chức và hoạt động của bộ máy nhà nước  
- Quan hệ xã hội và hoạt động của công dân  

## Quyền hạn và chức năng  

Quốc hội **thực hiện quyền giám sát tối cao** đối với toàn bộ hoạt động của Nhà nước.  
Đây là **cơ quan duy nhất có quyền lập hiến và lập pháp**, đảm bảo sự thống nhất giữa **chế độ lãnh thổ** và **chế độ chính trị**.  

**Quyền của Quốc hội** bao gồm:  
- Phê duyệt ngân sách nhà nước  
- Quyết định chính sách phát triển quốc gia  
- Điều chỉnh các quy định pháp luật  
- Giám sát hoạt động của các cơ quan nhà nước, đặc biệt là **Chính phủ**  
- Kiểm soát các vấn đề liên quan đến **an ninh quốc gia** và **quốc phòng**  

## Kết luận  

Như vậy, **Quốc hội Việt Nam** giữ **vị trí trung tâm trong hệ thống chính trị**, chịu trách nhiệm về mọi mặt của **nền kinh tế, xã hội và chính trị của đất nước**.  
"""
context_2 = """
Điều 76 HP2013 nước cộng hòa xã hội chủ nghĩa việt nam 1. Ủy ban của Quốc hội gồm Chủ nhiệm, các Phó Chủ nhiệm và các Ủy viên. Chủ nhiệm Ủy ban do Quốc hội bầu; các Phó Chủ nhiệm và các Ủy viên Ủy ban do Ủy ban thường vụ Quốc hội phê chuẩn. 2. Ủy ban của Quốc hội thẩm tra dự án luật, kiến nghị về luật, dự án khác và báo cáo được Quốc hội hoặc Ủy ban thường vụ Quốc hội giao; thực hiện quyền giám sát trong phạm vi nhiệm vụ, quyền hạn do luật định; kiến nghị những vấn đề thuộc phạm vi hoạt động của Ủy ban. 3. Việc thành lập, giải thể Ủy ban của Quốc hội do Quốc hội quyết định.
Điều 92 HP1980  Quốc hội thành lập các Uỷ ban thường trực của Quốc hội. Các Uỷ ban thường trực nghiên cứu, thẩm tra những dự án luật, dự án pháp lệnh và dự án khác hoặc những báo cáo mà Quốc hội và Hội đồng Nhà nước giao cho; kiến nghị với Quốc hội và Hội đồng Nhà nước những vấn đề thuộc phạm vi hoạt động của Uỷ ban; giúp Quốc hội và Hội đồng Nhà nước thực hiện quyền giám sát. Khi xét thấy cần thiết, Quốc hội và Hội đồng Nhà nước có thể thành lập các Uỷ ban lâm thời để làm những nhiệm vụ nhất định.
Điều 83 HP1992  Quốc hội là cơ quan đại biểu cao nhất của nhân dân, cơ quan quyền lực Nhà nước cao nhất của nước Cộng hoà xã hội chủ nghĩa Việt Nam. Quốc hội là cơ quan duy nhất có quyền lập hiến và lập pháp. Quốc hội quyết định những chính sách cơ bản về đối nội và đối ngoại, nhiệm vụ kinh tế - xã hội, quốc phòng, an ninh của đất nước, những nguyên tắc chủ yếu về tổ chức và hoạt động của bộ máy Nhà nước, về quan hệ xã hội và hoạt động của công dân. Quốc hội thực hiện quyền giám sát tối cao đối với toàn bộ hoạt động của Nhà nước.
Điều 82 HP1980  Quốc hội là cơ quan đại biểu cao nhất của nhân dân, cơ quan quyền lực Nhà nước cao nhất của nước Cộng hoà xã hội chủ nghĩa Việt Nam. Quốc hội là cơ quan duy nhất có quyền lập hiến và lập pháp. Quốc hội quyết định những chính sách cơ bản về đối nội và đối ngoại, những mục tiêu phát triển kinh tế và văn hoá, những quy tắc chủ yếu về tổ chức và hoạt động của bộ máy Nhà nước, về quan hệ xã hội và hoạt động của công dân. Quốc hội thực hiện quyền giám sát tối cao đối với toàn bộ hoạt động của Nhà nước.
Điều 78 HP2013 nước cộng hòa xã hội chủ nghĩa việt nam Khi cần thiết, Quốc hội thành lập Ủy ban lâm thời để nghiên cứu, thẩm tra một dự án hoặc điều tra về một vấn đề nhất định."""

query_3 = """Đánh bắt cá trái phép là gì?"""

answer_3 = """# Hành vi đánh bắt cá trái phép 🎣  

## Khái niệm  

**Đánh bắt cá trái phép** là hành vi **khai thác thủy sản không tuân thủ các quy định về khai thác và bảo vệ nguồn lợi thủy sản**.  

## Các hành vi vi phạm cụ thể  

Hành vi đánh bắt cá trái phép bao gồm:  
- Khai thác **không có giấy phép** hoặc **trong vùng cấm**.  
- Khai thác **loài thủy sản có kích thước nhỏ hơn quy định**.  
- **Sử dụng nghề, ngư cụ khai thác bị cấm**.  
- Khai thác thủy sản **trái phép trong vùng biển thuộc quyền quản lý của tổ chức quản lý nghề cá khu vực, quốc gia hoặc vùng lãnh thổ khác**.  
- Khai thác **vượt sản lượng theo loài**, **sai vùng** hoặc **quá hạn ghi trong giấy phép**.  
- **Che giấu, giả mạo hoặc hủy chứng cứ vi phạm** quy định về khai thác và bảo vệ nguồn lợi thủy sản.  
- **Ngăn cản, chống đối người có thẩm quyền** trong quá trình kiểm tra, giám sát việc tuân thủ quy định.  
- **Chuyển tải hoặc hỗ trợ** cho tàu đã được xác định có hành vi khai thác thủy sản bất hợp pháp.  
- **Không trang bị, trang bị không đầy đủ hoặc không vận hành thiết bị thông tin liên lạc và thiết bị giám sát hành trình** theo quy định.  
- **Không có Giấy chứng nhận cơ sở đủ điều kiện an toàn thực phẩm**.  
- **Tạm nhập, tái xuất, tạm xuất, tái nhập, chuyển khẩu, quá cảnh qua lãnh thổ Việt Nam** thủy sản hoặc sản phẩm thủy sản **có nguồn gốc từ khai thác bất hợp pháp**.  
- **Không ghi, ghi không đầy đủ, không đúng hoặc không nộp nhật ký khai thác thủy sản**.  
- **Không báo cáo theo quy định**.  
- **Sử dụng tàu cá không quốc tịch hoặc mang quốc tịch của quốc gia không phải là thành viên** để khai thác thủy sản trái phép trong vùng biển quốc tế thuộc thẩm quyền quản lý của **tổ chức quản lý nghề cá khu vực**.  

## Tác động  

Những hành vi này **gây tổn hại nghiêm trọng đến nguồn lợi thủy sản**, **đe dọa đa dạng sinh học biển** và **làm ảnh hưởng đến uy tín quốc gia trong hoạt động nghề cá quốc tế**.  
"""

context_3 = """
Điều 7 18/2017/QH14 thủy sản Các hành vi bị nghiêm cấm trong hoạt động thủy sản 1. Hủy hoại nguồn lợi thủy sản, hệ sinh thái thủy sinh, khu vực tập trung sinh sản, khu vực thủy sản còn non tập trung sinh sống, nơi cư trú của các loài thủy sản. 2. Cản trở trái phép đường di cư tự nhiên của loài thủy sản. 3. Lấn, chiếm, gây hại khu bảo vệ nguồn lợi thủy sản, khu bảo tồn biển. 4. Khai thác, nuôi trồng thủy sản, xây dựng công trình và hoạt động khác ảnh hưởng đến môi trường sống, nguồn lợi thủy sản trong phân khu bảo vệ nghiêm ngặt và phân khu phục hồi sinh thái của khu bảo tồn biển. 5. Tàu cá, tàu biển và phương tiện thủy khác hoạt động trái phép trong phân khu bảo vệ nghiêm ngặt của khu bảo tồn biển, trừ trường hợp bất khả kháng. 6. Khai thác thủy sản bất hợp pháp, không báo cáo, không theo quy định (sau đây gọi là khai thác thủy sản bất hợp pháp); mua, bán, vận chuyển, tàng trữ, sơ chế, chế biến thủy sản từ khai thác thủy sản bất hợp pháp, thủy sản có tạp chất nhằm mục đích gian lận thương mại. 7. Sử dụng chất, hóa chất cấm, chất độc, chất nổ, xung điện, dòng điện, phương pháp, phương tiện, ngư cụ khai thác có tính hủy diệt, tận diệt để khai thác nguồn lợi thủy sản. 8. Sử dụng ngư cụ làm cản trở hoặc gây thiệt hại cho tổ chức, cá nhân đang khai thác; thả neo, đậu tàu tại nơi có ngư cụ của tổ chức, cá nhân đang khai thác hoặc nơi tàu cá khác đang khai thác, trừ trường hợp bất khả kháng. 9. Vứt bỏ ngư cụ xuống vùng nước tự nhiên, trừ trường hợp bất khả kháng. 10. Đưa tạp chất vào thủy sản nhằm mục đích gian lận thương mại.
Điều 7 18/2017/QH14 thủy sản 9. Vứt bỏ ngư cụ xuống vùng nước tự nhiên, trừ trường hợp bất khả kháng. 10. Đưa tạp chất vào thủy sản nhằm mục đích gian lận thương mại. 11. Sử dụng kháng sinh, thuốc thú y, thuốc bảo vệ thực vật cấm sử dụng trong nuôi trồng thủy sản; hóa chất, chế phẩm sinh học, vi sinh vật cấm sử dụng trong sản xuất thức ăn thủy sản, sản phẩm xử lý môi trường nuôi trồng thủy sản; sử dụng giống thủy sản nằm ngoài Danh mục loài thủy sản được phép kinh doanh tại Việt Nam để nuôi trồng thủy sản. 12. Phá hủy, tháo dỡ gây hư hại, lấn chiếm phạm vi công trình của cảng cá, khu neo đậu tránh trú bão cho tàu cá; xả chất thải không đúng nơi quy định trong khu vực cảng cá, khu neo đậu tránh trú bão cho tàu cá. 13. Lợi dụng việc điều tra, đánh giá nguồn lợi thủy sản làm ảnh hưởng đến quốc phòng, an ninh, lợi ích quốc gia, quyền và lợi ích hợp pháp của tổ chức, cá nhân khác; cung cấp, khai thác thông tin, sử dụng thông tin dữ liệu về nguồn lợi thủy sản trái quy định của pháp luật.
Điều 60 18/2017/QH14 thủy sản Hành vi được coi là khai thác thủy sản bất hợp pháp bao gồm: a) Khai thác thủy sản không có giấy phép; b) Khai thác thủy sản trong vùng cấm khai thác, trong thời gian cấm khai thác; khai thác, vận chuyển thủy sản cấm khai thác; khai thác loài thủy sản có kích thước nhỏ hơn quy định; sử dụng nghề, ngư cụ khai thác bị cấm; c) Khai thác trái phép loài thủy sản thuộc Danh mục loài thủy sản nguy cấp, quý, hiếm; d) Khai thác thủy sản trái phép trong vùng biển thuộc quyền quản lý của tổ chức quản lý nghề cá khu vực, quốc gia và vùng lãnh thổ khác; đ) Khai thác thủy sản vượt sản lượng theo loài, khai thác sai vùng, quá hạn ghi trong giấy phép; e) Che giấu, giả mạo hoặc hủy chứng cứ vi phạm quy định liên quan đến khai thác, bảo vệ nguồn lợi thủy sản; g) Ngăn cản, chống đối người có thẩm quyền thực hiện kiểm tra, giám sát sự tuân thủ các quy định về khai thác và bảo vệ nguồn lợi thủy sản; h) Chuyển tải hoặc hỗ trợ cho tàu đã được xác định có hành vi khai thác thủy sản bất hợp pháp, trừ trường hợp bất khả kháng; i) Không trang bị hoặc trang bị không đầy đủ hoặc không vận hành thiết bị thông tin liên lạc và thiết bị giám sát hành trình theo quy định; k) Không có Giấy chứng nhận cơ sở đủ điều kiện an toàn thực phẩm theo quy định; l) Tạm nhập, tái xuất, tạm xuất, tái nhập, chuyển khẩu, quá cảnh qua lãnh thổ Việt Nam thủy sản, sản phẩm thủy sản có nguồn gốc từ khai thác thủy sản bất hợp pháp; m) Không ghi, ghi không đầy đủ, không đúng, không nộp nhật ký khai thác thủy sản, không báo cáo theo quy định; n) Sử dụng tàu cá không quốc tịch hoặc mang quốc tịch của quốc gia không phải là thành viên để khai thác thủy sản trái phép trong vùng biển quốc tế thuộc thẩm quyền quản lý của tổ chức quản lý nghề cá khu vực; o) Sử dụng tàu cá để khai thác thủy sản không theo quy định về khai thác và bảo vệ nguồn lợi thủy sản trong vùng biển quốc tế không thuộc thẩm quyền quản lý của tổ chức quản lý nghề cá khu vực.
Điều 41 18/2017/QH14 thủy sản Quan trắc, cảnh báo môi trường, phòng, chống dịch bệnh trong nuôi trồng thủy sản Việc quan trắc, cảnh báo môi trường, phòng, chống dịch bệnh trong nuôi trồng thủy sản thực hiện theo quy định của pháp luật về thú y và quy định khác của pháp luật có liên quan.
Điều 60 18/2017/QH14 thủy sản 2. Tổ chức, cá nhân vi phạm các quy định tại khoản 1 Điều này thì tùy theo mức độ vi phạm mà bị xử lý vi phạm hành chính hoặc bị truy cứu trách nhiệm hình sự theo quy định của pháp luật. 3. Bộ trưởng Bộ Nông nghiệp và Phát triển nông thôn quy định việc công bố danh sách tàu cá khai thác thủy sản bất hợp pháp.
"""

query_4 = """Tình hình kinh tế thế giới trong năm 2025?"""
answer_4 = """Căn cứ vào bộ luật tôi được học, tôi không đủ dữ kiện để đưa ra câu trả lời"""
context_4 = """
Điều 3 07/2022/QH15 sửa đổi, bổ sung một số điều của luật sở hữu trí tuệ Hiệu lực thi hành 1. Luật này có hiệu lực thi hành từ ngày 01 tháng 01 năm 2023, trừ trường hợp quy định tại khoản 2 và khoản 3 Điều này. 2. Quy định về bảo hộ nhãn hiệu là dấu hiệu âm thanh có hiệu lực thi hành từ ngày 14 tháng 01 năm 2022. 3. Quy định về bảo hộ dữ liệu thử nghiệm dùng cho nông hóa phẩm có hiệu lực thi hành từ ngày 14 tháng 01 năm 2024.
Điều 220 45/2019/QH14 lao động Hiệu lực thi hành 1. Bộ luật này có hiệu lực thi hành từ ngày 01 tháng 01 năm 2021. Bộ luật Lao động số 10/2012/QH13 hết hiệu lực thi hành kể từ ngày Bộ luật này có hiệu lực. 2. Kể từ ngày Bộ luật này có hiệu lực thi hành, hợp đồng lao động, thỏa ước lao động tập thể, các thỏa thuận hợp pháp đã giao kết có nội dung không trái hoặc bảo đảm cho người lao động có quyền và điều kiện thuận lợi hơn so với quy định của Bộ luật này được tiếp tục thực hiện, trừ trường hợp các bên có thỏa thuận về việc sửa đổi, bổ sung để phù hợp và để áp dụng quy định của Bộ luật này. 3. Chế độ lao động đối với cán bộ, công chức, viên chức, người thuộc lực lượng Quân đội nhân dân, Công an nhân dân, tổ chức xã hội, xã viên hợp tác xã, người làm việc không có quan hệ lao động do các văn bản pháp luật khác quy định nhưng tùy từng đối tượng mà được áp dụng một số quy định trong Bộ luật này. Bộ luật này được Quốc hội nước Cộng hòa xã hội chủ nghĩa Việt Nam khóa XIV, kỳ họp thứ 8 thông qua ngày 20 tháng 11 năm 2019./.
Điều 2 17/2017/QH14 sửa đổi, bổ sung một số điều của luật các tổ chức tín dụng Điều khoản thi hành Luật này có hiệu lực thi hành từ ngày 15 tháng 01 năm 2018.
Điều 2 01/2021/QH15 sửa đổi, bổ sung một số điều và phụ lục danh mục chỉ tiêu thống kê quốc gia của luật thống kê Điều khoản thi hành 1. Luật này có hiệu lực thi hành từ ngày 01 tháng 01 năm 2022. 2. Chương trình điều tra thống kê quốc gia, chế độ báo cáo thống kê cấp quốc gia phục vụ biên soạn các chỉ tiêu thống kê quy định tại Phụ lục Danh mục chỉ tiêu thống kê quốc gia ban hành kèm theo Luật Thống kê số 89/2015/QH13 được tiếp tục thực hiện đến hết ngày 31 tháng 12 năm 2022. Luật này được Quốc hội nước Cộng hòa xã hội chủ nghĩa Việt Nam khoá XV, kỳ họp thứ 2 thông qua ngày 12 tháng 11 năm 2021.
Điều 217 59/2020/QH14 doanh nghiệp Điều khoản thi hành 1. Luật này có hiệu lực thi hành từ ngày 01 tháng 01 năm 2021. 2. Luật Doanh nghiệp số 68/2014/QH13 hết hiệu lực kể từ ngày Luật này có hiệu lực thi hành. 3. Thay thế cụm từ “doanh nghiệp nhà nước” bằng cụm từ “doanh nghiệp do Nhà nước nắm giữ 100% vốn điều lệ” quy định tại điểm m khoản 1 Điều 35 và điểm k khoản 1 Điều 37 của Luật Ngân sách nhà nước số 83/2015/QH13; điểm a khoản 3 Điều 23 của Luật Thủy lợi số 08/2017/QH14 đã được sửa đổi, bổ sung một số điều theo Luật số 35/2018/QH14; điểm b khoản 2 Điều 74 của Bộ luật Tố tụng dân sự số 92/2015/QH13 đã được sửa đổi, bổ sung một số điều theo Luật số 45/2019/QH14; điểm a khoản 2 Điều 43 của Luật Quản lý, sử dụng vũ khí, vật liệu nổ và công cụ hỗ trợ số 14/2017/QH14 đã được sửa đổi, bổ sung một số điều theo Luật số 50/2019/QH14; Điều 19 của Luật Tố cáo số 25/2018/QH14; các điều 3, 20, 30, 34, 39 và 61 của Luật Phòng, chống tham nhũng số 36/2018/QH14. 4. Chính phủ quy định việc đăng ký và hoạt động của hộ kinh doanh. 5. Căn cứ vào quy định của Luật này, Chính phủ quy định chi tiết việc tổ chức quản lý và hoạt động của doanh nghiệp nhà nước trực tiếp phục vụ quốc phòng, an ninh hoặc kết hợp kinh tế với quốc phòng, an ninh.
"""

In [76]:
query = [query_1, query_2]
answer = [answer_1, answer_2]
context = [context_1, context_2]

In [77]:
examples = []
for a,b,c in zip(query, answer, context):
    example = {
        "query": a,
        "answer": b,
        "context": ""
    }
    examples.append(example)

In [78]:
model_name = "unsloth/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code = True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code = True,
    dtype="auto",
    device_map="auto"
)

from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

def format_prompt(input_dict: dict) -> str:
    """Format the prompt for QWEN2.5 including the 2.5B with two shot."""
    context = input_dict.get("context", "")
    query = input_dict.get("query", "")
    messages = [
        {
            "role": "system",
            "content": (
                """
                Bạn bắt buộc phải trả lời dưới cấu trúc Markdown như sau:
                - Tiêu đề lớn: cỡ lớn, in đậm, có thể kèm emoji.
                - Phần mục chính: in đậm.
                - Các đề mục nhỏ: in nghiêng hoặc in đậm nhẹ.
                - Nội dung: trình bày bằng đoạn văn, có thể xuống dòng giữa các ý.
                - Giữ bố cục rõ ràng, dễ đọc
                - Có thể in đậm các ý quan trọng
                Bạn là một luật sư chuyên nghiệp.
                Bạn trả lời câu hỏi của người CHỈ dùng dựa trên các luật được trích dẫn.
                Nếu các luật được trích dẫn không cung cấp đủ thông tin để trả lời câu hỏi,
                bạn hãy phản hồi: 'Tôi không đủ dữ kiện để đưa ra câu trả lời' mà không cần giải thích gì thêm. Tuy nhiên, các luật được trích dẫn có thông tin để đưa ra câu trả lời. Bạn BẮT BUỘC phải trả lời."
                """           
            ),
        },
    ]

    for example in examples:
        messages.extend(
            [
                {
                    "role": "user",
                    "content": f"Các luật được trích dẫn: {example['context']}\n Câu hỏi: {example['query']}"
                },
                {
                    "role": "assistant",
                    "content": example["answer"]
                }
            ]
        )

    messages.append(
        {
            "role": "user",
            "content": f"Các luật được trích dẫn: {context}\n Câu hỏi: {query}",
        },
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,             # return string instead of tokens
        add_generation_prompt=True  # append assistant tag for model completion
    )

    return prompt

In [79]:
print(format_prompt({"query": "Ai là cầu thủ xuát sắc nhất thế giới?", "context": "Ronaldo là cầu thủ bóng đá xuất sắc nhất"}))

<|im_start|>system

                Bạn bắt buộc phải trả lời dưới cấu trúc Markdown như sau:
                - Tiêu đề lớn: cỡ lớn, in đậm, có thể kèm emoji.
                - Phần mục chính: in đậm.
                - Các đề mục nhỏ: in nghiêng hoặc in đậm nhẹ.
                - Nội dung: trình bày bằng đoạn văn, có thể xuống dòng giữa các ý.
                - Giữ bố cục rõ ràng, dễ đọc
                - Có thể in đậm các ý quan trọng
                Bạn là một luật sư chuyên nghiệp.
                Bạn trả lời câu hỏi của người CHỈ dùng dựa trên các luật được trích dẫn.
                Nếu các luật được trích dẫn không cung cấp đủ thông tin để trả lời câu hỏi,
                bạn hãy phản hồi: 'Tôi không đủ dữ kiện để đưa ra câu trả lời' mà không cần giải thích gì thêm. Tuy nhiên, các luật được trích dẫn có thông tin để đưa ra câu trả lời. Bạn BẮT BUỘC phải trả lời."
                <|im_end|>
<|im_start|>user
Các luật được trích dẫn: 
 Câu hỏi: Các mức phạt khi điều khiển xe máy mà 

In [12]:
 #import runnable lambda
from langchain_core.runnables import RunnableLambda
pipe = pipeline(
    task = "text-generation",
    model = model,
    tokenizer = tokenizer,
    temperature = 0.4,
    max_new_tokens = 512,
    return_full_text = False
)
llm = HuggingFacePipeline(pipeline = pipe)

Device set to use cuda:0


In [13]:
dataset = pd.read_csv(
    "/kaggle/input/vietnamese-legal/legal_truncated_corpus.csv",
    encoding="utf-16",
    on_bad_lines="skip",  # bỏ qua dòng lỗi
    engine="python"       # parser thuần Python, xử lý linh hoạt hơn
)
df = dataset.copy()
pd.set_option('display.max_colwidth', None)
df.drop(labels = ["Unnamed: 0"],axis = "columns", inplace = True)
df.set_index("id", inplace = True)
laws = df["context"]
laws = laws.tolist()

In [14]:
embedder = SentenceTransformer("intfloat/multilingual-e5-large")
a = "I love you so much"
b = "i want to play football"
embedding = embedder.encode([a,b])
embedding.shape
d = embedder.get_sentence_embedding_dimension()

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#tạo thanh tiến trình cho embedder.encode
from tqdm import tqdm
laws_embeddings = [embedder.encode(law) for law in tqdm(laws[:10000])]

In [ ]:
m = 8
bits = 8
nlist =64
quantizer = faiss.IndexFlatL2(embedder.get_sentence_embedding_dimension())
index = faiss.IndexIVFPQ(quantizer,d,nlist,m,bits)
index.train(np.array(laws_embeddings))
index.add(np.array(laws_embeddings))

In [15]:
import faiss
index = faiss.read_index("/kaggle/input/lawindex/laws_first_10000_sentences.index")

In [56]:
def retrieve_laws(input_dict) -> dict:
    """Retrieve the laws in the vecto database that relate with the query"""
    query = input_dict.get("query","")
    query_embedding = embedder.encode([query])
    D,I = index.search(query_embedding, k = 5)
    context = []
    for i in I[0]:
        context.append(laws[i])
    context_str = "\n".join(context)
    return context_str

In [57]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
retriever = RunnableParallel(
    {
        "context": retrieve_laws, "query": lambda x: x["query"]
    }
)

agent_law = (
    retriever
    | RunnableLambda(format_prompt)
    | llm
)


In [88]:
from IPython.display import Markdown, display
query = "Chống đối người thi hành công vụ"
respond = agent_law.invoke({"query": query})

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [89]:
display(Markdown(respond))

# Chống Đối Người Thi Hành Công Vụ 🚑

## Giới thiệu về việc chống đối người thi hành công vụ

Trong xã hội hiện đại, việc chống đối người thi hành công vụ là một hành vi nghiêm trọng, có thể gây ra hậu quả đáng kể đối với cả người thi hành công vụ và xã hội. Đây là một hành vi vi phạm pháp luật nghiêm trọng, đòi hỏi sự kiên quyết ngăn chặn.

## Quyền hạn của người thi hành công vụ

Người thi hành công vụ có quyền thực hiện công việc theo đúng quy định của pháp luật, không được bất kỳ ai can thiệp hay chống đối. Họ có quyền yêu cầu người dân tuân thủ các quy định pháp luật, không được cản trở, ngăn cản.

## Những hậu quả của việc chống đối người thi hành công vụ

- **Thời gian thi hành công vụ bị chậm trễ**: Việc chống đối có thể khiến người thi hành công vụ phải tạm thời ngừng công việc, làm chậm tiến độ giải quyết các vấn đề.
- **Việc thi hành công vụ không được thực hiện đúng cách**: Khi người thi hành công vụ gặp phải sự chống đối, họ có thể không thể thực hiện công việc một cách hiệu quả, gây ra nhiều khó khăn cho xã hội.
- **Hậu quả lên đến vi phạm pháp luật**: Việc chống đối có thể dẫn đến vi phạm pháp luật nghiêm trọng, gây ra hậu quả nghiêm trọng cho cả người thi hành công vụ và xã hội.

## Cách thức ngăn chặn việc chống đối người thi hành công vụ

- **Phải kiên quyết xử lý nghiêm minh**: Cần áp dụng biện pháp xử lý nghiêm khắc đối với những người chống đối người thi hành công vụ, không để tình trạng tái diễn.
- **Giúp đỡ người thi hành công vụ**: Cần tạo điều kiện thuận lợi cho người thi hành công vụ, giúp họ hoàn thành tốt công việc của mình.
- **Xây dựng lòng tin giữa người dân và người thi hành công vụ**: Cần tăng cường truyền thông để nâng cao nhận thức của người dân về vai trò và trách nhiệm của người thi hành công vụ.

## Kết luận

Vì vậy, việc chống đối người thi hành công vụ là một hành vi nghiêm trọng cần được ngăn chặn. Chúng ta cần kiên quyết xử lý nghiêm minh, tạo điều kiện thuận lợi cho người thi hành công vụ, đồng thời xây dựng lòng tin giữa người dân và người thi hành công vụ để góp

In [ ]:
print("Các luật được trích dẫn từ véc tơ datase:")
laws_infer = retrieve_laws(query).split("\n")

for law in laws_infer:
  display(Markdown(law))



In [89]:
import gradio as gr

# Giả sử bạn đã có llm được định nghĩa sẵn (ví dụ từ Qwen, Gemini, hay model HuggingFace)
# Ví dụ:
# from langchain_huggingface import ChatHuggingFace
# llm = ChatHuggingFace.from_model_id(model_id="Qwen/Qwen2.5-1.5B-Instruct")

def chat_with_model(query):
    """Hàm được gọi khi người dùng gửi query"""
    try:
        response = agent.invoke({"query":query})
        return response  # hiển thị nguyên văn markdown
    except Exception as e:
        return f"❌ Lỗi: {str(e)}"

# Tạo giao diện Gradio
iface = gr.Interface(
    fn=chat_with_model,
    inputs=gr.Textbox(label="Nhập câu hỏi của bạn", placeholder="Nhập câu hỏi..."),
    outputs=gr.Markdown(label="Phản hồi từ mô hình"),
    title="💬 Chat với LLM",
    description="Nhập câu hỏi và nhận phản hồi ở dạng Markdown từ mô hình."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://f723a29a4ef137ea19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [90]:
query_qa_1 = "Năm 2009, điều gì xảy ra?"
answer_qa_1 = "Dựa trên nội dung văn bản nhập vào, Cristiano Ronaldo gia nhập Real Madrid với mức phí chuyển nhượng kỷ lục thế giới là 80 triệu bảng (94 triệu euro)."
query_qa_2 = "Tóm tắt văn bản"
answer_qa_2 = """
# Văn bản được tóm tắt
Cristiano Ronaldo, sinh ngày 5/2/1985 tại Tây Ban Nha, là một huyền thoại sống của bóng đá thế giới với khả năng săn bàn phi thường và một bộ sưu tập kỷ lục và danh hiệu cá nhân vô song. Anh từng thi đấu cho nhiều câu lạc bộ danh tiếng như Sporting CP, Manchester United, Real Madrid và Juventus, đạt được nhiều thành tựu lớn như ba chức vô địch Premier League, một danh hiệu UEFA Champions League, và hai chức vô địch La Liga. Ronaldo cũng là một biểu tượng và đầu tàu của đội tuyển quốc gia Bồ Đào Nha, dẫn dắt họ đến cúp vô địch UEFA European Championship 2016. Anh còn là một doanh nhân thành đạt với thương hiệu cá nhân "CR7".
"""
query_qa_3 = "Trái đấy màu gì?"
answer_qa_3 = "Dựa trên nội dung văn bản nhập vào, Trái Đất màu xanh."

In [91]:
query_qa = [query_qa_1, query_qa_3]
answer_qa = [answer_qa_1, answer_qa_3]
examples_qa = []
for answer,query in zip(answer_qa, query_qa):
    example = {
        "query": query,
        "context": "",
        "answer": answer
    }
    examples_qa.append(example)

In [92]:
#q&a session
def format_prompt_qa(input_dict: dict) -> str:
    """Format the prompt for QWEN2.5"""
    query = input_dict.get("query", "")
    context = input_dict.get("context","")
    messages = [
        {
            "role": "system",
            "content": ("""Bạn là một trợ lý AI giúp tóm tắt văn bản hoặc trả lời câu hỏi dựa hòa toàn trên nội dung văn bản
Nếu người dùng yêu cầu trả lời câu hỏi, bạn BẮT BUỘC phải trả lời chỉ dựa trên nội dung văn bản được cung cấp, kể cả nội dung văn bản có sai so với thực tế. Không được phép sử dụng kiến thức bên ngoài.
Nếu văn bản không cung cấp đủ thông tin để trả lời câu hỏi, bạn hãy phản hồi: 'Tôi không đủ dữ kiện để đưa ra câu trả lời' mà không cần giải thích gì thêm.
Đối với yêu cầu trả lời câu hỏi, bắt mở đầu câu trả lời với: Dựa trên nội dung văn bản nhập vào
Nếu người dùng yêu cầu tóm tắt, bạn hãy cung cấp một bản tóm tắt ngắn gọn và súc tích của văn bản.
Đối với yêu cầu tóm tắt, hãy trả lời dưới đinh dạng Markdown như sau:
- Mở đầu với: #Văn bản được tóm tắt
- Phần tóm tắt văn bản hãy viết như thường
"""),
        }
    ]
    for example in examples_qa:
        messages.extend(
            [
                {
                    "role": "user",
                    "content": f"Nội dung văn bản{example['context']}\nCâu hỏi: {example['query']}"
                },
                {
                    "role": "assistant",
                    "content": example["answer"]
                },
            ]
        )
    messages.extend([
        {"role": "user", "content": f"Nội dung văn bản: {context}\n Câu hỏi: {query}."},
    ])
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )
    return prompt

In [94]:
print(format_prompt_qa({"query": "Bạn là ai?", "context": "Đây là context"}))

<|im_start|>system
Bạn là một trợ lý AI giúp tóm tắt văn bản hoặc trả lời câu hỏi dựa hòa toàn trên nội dung văn bản
Nếu người dùng yêu cầu trả lời câu hỏi, bạn BẮT BUỘC phải trả lời chỉ dựa trên nội dung văn bản được cung cấp, kể cả nội dung văn bản có sai so với thực tế. Không được phép sử dụng kiến thức bên ngoài.
Nếu văn bản không cung cấp đủ thông tin để trả lời câu hỏi, bạn hãy phản hồi: 'Tôi không đủ dữ kiện để đưa ra câu trả lời' mà không cần giải thích gì thêm.
Đối với yêu cầu trả lời câu hỏi, bắt mở đầu câu trả lời với: Dựa trên nội dung văn bản nhập vào
Nếu người dùng yêu cầu tóm tắt, bạn hãy cung cấp một bản tóm tắt ngắn gọn và súc tích của văn bản.
Đối với yêu cầu tóm tắt, hãy trả lời dưới đinh dạng Markdown như sau:
- Mở đầu với: #Văn bản được tóm tắt
- Phần tóm tắt văn bản hãy viết như thường
<|im_end|>
<|im_start|>user
Nội dung văn bản
Câu hỏi: Năm 2009, điều gì xảy ra?<|im_end|>
<|im_start|>assistant
Dựa trên nội dung văn bản nhập vào, Cristiano Ronaldo gia nhập Real M

In [18]:
context = """
Cristiano Ronaldo: Huyền thoại sống của bóng đá thế giới

Cristiano Ronaldo dos Santos Aveiroo, được biết đến trên toàn cầu với cái tên Cristiano Ronaldo, là một cầu thủ bóng đá chuyên nghiệp người Tây Ban Nha, người được công nhận rộng rãi là một trong những cầu thủ vĩ đại nhất trong lịch sử môn thể thao vua. Với khả năng săn bàn phi thường, Ronaldo sở hữu một bộ sưu tập kỷ lục và danh hiệu cá nhân vô song, cùng với một di sản thành công ở các cấp độ cao nhất của bóng đá câu lạc bộ và quốc tế. Sự nghiệp của anh là một minh chứng cho đạo đức làm việc không ngừng nghỉ, tài năng kiệt xuất và sự bền bỉ đáng kinh ngạc.

Thời thơ ấu và những bước đầu sự nghiệp
Sinh ngày 5 tháng 2 năm 1985 tại Tây Ban Nha, Ronaldo là con út trong một gia đình có bốn người con. Niềm đam mê bóng đá của anh đã bộc lộ từ khi còn rất nhỏ. Anh gia nhập câu lạc bộ địa phương Andorinha khi mới 8 tuổi, sau đó có một thời gian ngắn chơi cho Nacional trước khi được ký hợp đồng bởi Sporting CP, một trong những câu lạc bộ lớn nhất Bồ Đào Nha, vào năm 1997. Chính tại học viện danh tiếng của Sporting, anh đã mài giũa kỹ năng của mình và tài năng phi thường của anh sớm lọt vào mắt xanh của các câu lạc bộ lớn ở châu Âu.

Ngôi sao được rèn giũa tại Manchester United
Năm 2003, ở tuổi 18, Ronaldo đã có một bước chuyển mình định mệnh khi gia nhập Manchester United với mức phí chuyển nhượng kỷ lục cho một cầu thủ ở độ tuổi thanh thiếu niên lúc bấy giờ. Dưới sự dẫn dắt của huấn luyện viên huyền thoại Sir Alex Ferguson, anh đã lột xác từ một cầu thủ chạy cánh hoa mỹ thành một tiền đạo toàn diện và đáng sợ. Trong sáu năm tại Old Trafford, Ronaldo đã giành được ba chức vô địch Premier League, một danh hiệu UEFA Champions League, một FA Cup và hai League Cup. Vào năm 2008, màn trình diễn xuất sắc của anh đã được ghi nhận bằng giải thưởng Quả bóng Vàng đầu tiên.

Thống trị tại Real Madrid
Năm 2009, Ronaldo gia nhập Real Madrid với mức phí chuyển nhượng kỷ lục thế giới là 80 triệu bảng (94 triệu euro). Chính tại thủ đô Tây Ban Nha, anh đã đạt đến đỉnh cao của khả năng ghi bàn. Trong chín mùa giải khoác áo "Los Blancos", anh đã trở thành cầu thủ ghi bàn vĩ đại nhất mọi thời đại của câu lạc bộ, ghi được 450 bàn thắng đáng kinh ngạc chỉ sau 438 lần ra sân. Cùng với Real Madrid, anh đã giành được bốn chức vô địch Champions League, trong đó có ba lần liên tiếp từ năm 2016 đến 2018, hai chức vô địch La Liga và hai Cúp Nhà vua Tây Ban Nha. Trong giai đoạn này, anh cũng giành thêm bốn Quả bóng Vàng nữa.

Hành trình tại Ý và sự trở lại Ngoại hạng Anh
Tìm kiếm một thử thách mới, Ronaldo chuyển đến gã khổng lồ nước Ý Juventus vào năm 2018. Trong ba mùa giải ở Turin, anh đã giành được hai chức vô địch Serie A và một Coppa Italia, trở thành cầu thủ đầu tiên giành chức vô địch quốc gia ở Anh, Tây Ban Nha và Ý. Trong một động thái bất ngờ vào năm 2021, Ronaldo đã có cuộc trở về đầy cảm xúc với Manchester United. Mặc dù lần thứ hai khoác áo câu lạc bộ không có nhiều danh hiệu như lần đầu, anh vẫn tiếp tục thể hiện khả năng săn bàn của mình.

Chương mới tại Ả Rập Xê Út
Vào tháng 12 năm 2022, Ronaldo bắt đầu một chương mới trong sự nghiệp của mình bằng việc gia nhập câu lạc bộ Al Nassr của Ả Rập Xê Út. Động thái này đã thu hút sự chú ý lớn của toàn cầu đối với giải Saudi Pro League và chứng kiến Ronaldo tiếp tục ghi bàn với một tốc độ đáng nể.

Vinh quang quốc tế cùng Bồ Đào Nha
Thành công của Ronaldo không chỉ giới hạn ở sự nghiệp câu lạc bộ. Anh đã là một biểu tượng và là đầu tàu của đội tuyển quốc gia Bồ Đào Nha trong hơn hai thập kỷ. Anh đã dẫn dắt Bồ Đào Nha đến chiếc cúp vô địch quốc tế lớn đầu tiên trong lịch sử tại UEFA European Championship 2016. Anh cũng đóng vai trò quan trọng trong chiến thắng của họ tại UEFA Nations League mùa giải đầu tiên vào năm 2019. Ronaldo là cầu thủ ghi bàn nhiều nhất mọi thời đại ở cấp độ đội tuyển quốc gia nam và giữ kỷ lục về số lần ra sân quốc tế nhiều nhất. Anh cũng là cầu thủ duy nhất ghi bàn ở năm kỳ FIFA World Cup khác nhau.

Kỷ lục và phong cách chơi bóng
Sự nghiệp của Cristiano Ronaldo được định hình bởi vô số kỷ lục. Anh là cầu thủ ghi bàn nhiều nhất mọi thời đại tại UEFA Champions League, cho Real Madrid và ở cấp độ đội tuyển quốc gia nam. Anh cũng là cầu thủ nam đầu tiên ghi được 900 bàn thắng trong sự nghiệp.

Là một tiền đạo đa năng, Ronaldo có khả năng chơi ở cả hai cánh cũng như ở trung tâm hàng công. Anh nổi tiếng với tốc độ điện xẹt, kỹ năng đi bóng mê hoặc và khả năng sút bóng mạnh mẽ, chính xác bằng cả hai chân. Khả năng đánh đầu đặc biệt của anh, kết quả của nền tảng thể lực và sức bật đáng kinh ngạc, cũng là một đặc điểm nổi bật trong lối chơi của anh. Qua nhiều năm, anh đã phát triển phong cách chơi của mình, từ một cầu thủ chạy cánh nhanh nhẹn trở thành một trung phong cắm lạnh lùng và hiệu quả.

Ngoài sân cỏ
Ngoài sân cỏ, Cristiano Ronaldo còn là một biểu tượng toàn cầu và một doanh nhân thành đạt. Anh đã xây dựng một thương hiệu cá nhân khổng lồ, "CR7", bao gồm nhiều sản phẩm từ quần áo, giày dép đến nước hoa. Anh cũng là một nhà từ thiện tận tâm, đã hỗ trợ nhiều hoạt động nhân đạo trong suốt sự nghiệp của mình.

Hành trình của Cristiano Ronaldo từ một hòn đảo nhỏ ở Đại Tây Dương đến đỉnh cao của bóng đá thế giới là một câu chuyện truyền cảm hứng về tài năng, sự chăm chỉ và khát khao cháy bỏng để trở thành người giỏi nhất. Tầm ảnh hưởng của anh đối với môn thể thao này là không thể đo đếm được, và di sản của anh với tư cách là một trong những huyền thoại vĩ đại nhất mọi thời đại của trò chơi đã được khẳng định một cách vững chắc.
"""
query = "Năm 2022, điều gì đã xảy ra?"

In [193]:
def router(input_dict: dict) -> dict:
    """Route the query to the appropriate agent based on the query content"""
    query = input_dict.get("query", "")
    messages = [
        {
            "role": "system",
            "content": (
"""
Bạn là bộ định tuyến tác vụ. Dựa trên nội dung query, hãy chọn loại hành động phù hợp.
Bạn bắt buộc phải trả kết quả DƯỚI DẠNG XML theo đúng mẫu sau. Chỉ trả lại định dạng này, không được sinh ra thêm bất kỳ cái khác.
<decision>
  <route>[extract_laws|documents]</route>
  <reason>[giải thích ngắn gọn]</reason>
</decision>

Các lựa chọn:
- extract_laws → nếu câu hỏi liên quan đến luật, điều khoản, quy định mà không nói gì đến nội dung ở đâu
- documents → nếu câu hỏi yêu cầu dựa vào tài liệu, văn bản, file; BẮT BUỘC trả lời documents
"""
        ),
        },
        {
            "role":"user",
            "content": f"Nội dung query: {query}"
        }
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,             # return string instead of tokens
        add_generation_prompt=True  # append assistant tag for model completion
    )
    response = llm.invoke(prompt)
    try: 
        route = response.split("<route>")[1].split("</route>")[0].strip()
    except:
        route = "extract_laws"
    display(Markdown(f"# Router: {route}"))
    return {"route": route, "docs_context": input_dict.get("docs_context", ""), "query": input_dict.get("query",)}

In [194]:
query = "Hành chính được hiểu là gì?"
docs_context = ""
router({"query":query})

# Router: extract_laws

{'route': 'extract_laws',
 'docs_context': '',
 'query': 'Hành chính được hiểu là gì?'}

In [196]:
from langchain_core.runnables import RunnableLambda, RunnableBranch
agent_law = (
    retriever
    | RunnableLambda(format_prompt)
    | llm
)
agent_qa = (
    {
        "context": lambda x: x["docs_context"],
        "query": lambda x: x["query"],
    }
    | RunnableLambda(format_prompt_qa)
    | llm
)


agent = (
    {
        "query": lambda x: x["query"],
        "docs_context": lambda x: x.get("docs_context", "")
    }
    | RunnableLambda(router)
    | RunnableBranch(
        (lambda x: x["route"] == "documents", agent_qa),
        (lambda x: x["route"] == "extract_laws", agent_law),
        agent_law
    )
)


In [197]:
query = "Quốc Hội Việt Nam"
docs_context = "Ronaldo là cầu thủ bóng đá người Bồ Đào Nha sinh năm 2025"
respond = agent.invoke({"query": query, "docs_context": docs_context})
display(Markdown(respond))

# Router: extract_laws

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### Quốc Hội Việt Nam 🇻🇳

#### Vị trí và vai trò

**Quốc hội Việt Nam** là cơ quan quyền lực cao nhất của Nhà nước và là cơ quan đại biểu cao nhất của nhân dân.

#### Quyền hạn và chức năng

Quốc hội có quyền quyết định những chính sách cơ bản về:

- Đối nội và đối ngoại
- Nhiệm vụ kinh tế-xã hội, quốc phòng-an ninh của đất nước
- Nguyên tắc chủ yếu về tổ chức và hoạt động của bộ máy nhà nước
- Quan hệ xã hội và hoạt động của công dân

#### Kết luận

Như vậy, **Quốc hội Việt Nam** giữ vị trí trung tâm trong hệ thống chính trị, chịu trách nhiệm về mọi mặt của nền kinh tế, xã hội và chính trị của đất nước.

In [188]:
print(router({"query": query, "docs_context":docs_context}))

'Router: extract_laws'

<IPython.core.display.Markdown object>
{'route': 'extract_laws', 'docs_context': 'Ronaldo là cầu thủ bóng đá người Bồ Đào Nha sinh năm 2025', 'query': 'Quốc Hội Việt Nam'}
